In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [3]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [4]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [5]:
df.shape

(898, 35)

In [6]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
type(X_test_scaled)

numpy.ndarray

In [11]:
type(y_test)

numpy.ndarray

# ANN Classification

In [12]:
import torch
import torch.nn as nn

In [13]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [14]:
# multiclass classification so use long
# CrossEntropyloss needs yb in long format

In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [16]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            #1
            nn.Linear(X_train.shape[1], 64),
            #nn.Linear(X_pca.shape[1], 64),
            nn.ReLU(),

            #2
            nn.Linear(64, 64),
            nn.ReLU(),

            #output
            nn.Linear(64, 7),
        )
    def forward(self,x):
        return self.model(x)

In [17]:
import torch.optim as optim

model = ANN()

crie = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [18]:
#train_losses = []
#val_losses = []
#best_val_loss = float('inf') 

epochs = 100

for epoch in range(epochs):
    
    model.train()
    
    running_loss = 0.0

    for xb,yb in train_loader:
        
        optimizer.zero_grad() #optimizers bydefault accumulate prev gradient values -> but we don't want that
        
        outputs = model(xb)
        
        loss = crie(outputs, yb)
        
        loss.backward()
        
        optimizer.step() #param update step

        running_loss += loss.item() #tensor to float

    epoch_train_loss = running_loss/len(train_loader) #avg loss per batch for 1 epoch

    #train_losses.append(epoch_train_loss)

    print(f"For {epoch+1}/{epochs} ==> Train loss = {epoch_train_loss:.4f}")

For 1/100 ==> Train loss = 1.7479
For 2/100 ==> Train loss = 1.1274
For 3/100 ==> Train loss = 0.6863
For 4/100 ==> Train loss = 0.5283
For 5/100 ==> Train loss = 0.4348
For 6/100 ==> Train loss = 0.3790
For 7/100 ==> Train loss = 0.3230
For 8/100 ==> Train loss = 0.2932
For 9/100 ==> Train loss = 0.2609
For 10/100 ==> Train loss = 0.2421
For 11/100 ==> Train loss = 0.2360
For 12/100 ==> Train loss = 0.2146
For 13/100 ==> Train loss = 0.2094
For 14/100 ==> Train loss = 0.2121
For 15/100 ==> Train loss = 0.2021
For 16/100 ==> Train loss = 0.1855
For 17/100 ==> Train loss = 0.1758
For 18/100 ==> Train loss = 0.1697
For 19/100 ==> Train loss = 0.1690
For 20/100 ==> Train loss = 0.1568
For 21/100 ==> Train loss = 0.1558
For 22/100 ==> Train loss = 0.1552
For 23/100 ==> Train loss = 0.1487
For 24/100 ==> Train loss = 0.1433
For 25/100 ==> Train loss = 0.1397
For 26/100 ==> Train loss = 0.1378
For 27/100 ==> Train loss = 0.1290
For 28/100 ==> Train loss = 0.1358
For 29/100 ==> Train loss = 0

In [19]:
model.eval()

total = 0
correct = 0
#running_val_loss = 0.0

with torch.no_grad():
    for xb,yb in test_loader:
            
        outputs = model(xb) #[0.2, 0.5, 1.3...] - 7 vals

        _, predicted =  torch.max(outputs, 1) #returns max and index, max we don't want to save that's why _

        correct += (predicted == yb).sum().item()
        total += yb.size(0) #actual samples in each batch

print("total = ", total)
print("correct = ", correct)
print("Accuracy = ", correct/total)

total =  180
correct =  167
Accuracy =  0.9277777777777778


In [20]:
from sklearn.metrics import accuracy_score
model.eval()
with torch.no_grad():
    train_pred = model(X_train_tensor)
    test_pred = model(X_test_tensor)
    
    train_ce_loss = crie(train_pred, y_train_tensor)
    test_ce_loss = crie(test_pred, y_test_tensor)

print("Training Accuracy: ", accuracy_score(torch.argmax(train_pred, dim=1), y_train_tensor.numpy()))
print("Testing Accuracy: ", accuracy_score(torch.argmax(test_pred, dim=1), y_test_tensor.numpy()))

Training Accuracy:  0.9958217270194986
Testing Accuracy:  0.9277777777777778


In [35]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 4)
X_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.fit_transform(X_test_scaled)

In [36]:
X_train_tensor = torch.tensor(X_pca, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_pca, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)